# NeuroMAS: Train a Network of LLM Agents from One Final Reward

## Overview

Most multi-agent systems are engineered by hand: someone writes a planner prompt, a critic prompt, a verifier prompt, and wires them together. **NeuroMAS** takes a different route. It treats the whole multi-agent system as a *neural network whose neurons are language models*, and trains every agent jointly from a single signal: whether the final answer was correct.

In this tutorial you will build NeuroMAS from scratch in plain PyTorch and learn how to:

- Run many trainable agents on **one frozen backbone** by giving each agent its own small LoRA adapter
- Write **role-free, structure-aware** prompts, where an agent knows only its position in the network
- Route **private messages** between layers with a deterministic parser
- Train all agents with **REINFORCE from one shared terminal reward**, with no intermediate labels or per-agent rewards
- **Grow** a trained small network into a larger one, the way you might widen a neural network

The tutorial implements the method from [NeuroMAS: Multi-Agent Systems as Neural Networks with Joint Reinforcement Learning](https://arxiv.org/abs/2605.16757) (STAI-X 2026, Paper Award Honorable Mention). The reference code is at [haoranlustat/NeuroMAS](https://github.com/haoranlustat/NeuroMAS).

## Detailed Explanation

### Motivation

Hand-designed roles are a strong assumption. They fix what each agent should do before the system has seen any data, and scaling them means writing more prompts. Neural networks solved the same problem decades ago: nobody tells a hidden unit what to detect. It learns its function because the loss flows back to it.

NeuroMAS applies that idea to agents. The analogy is direct:

| Neural network | NeuroMAS |
|---|---|
| Neuron | Language-agent node (a backbone plus its own LoRA adapter) |
| Activation value on an edge | Text message sent to a downstream node |
| Layer | A group of nodes that read the previous layer's messages |
| Output layer | An output aggregation node that produces the answer |
| Loss | One terminal reward: was the final answer correct? |
| Backpropagation | REINFORCE, since text is discrete and cannot be differentiated through |
| Widening or deepening a trained net | Progressive growth: NeuroMAS-3, then -5, then -7 |

### The agent's state, tools, and control flow

- **State.** For one query, the state is the set of messages produced so far, plus each node's log-probabilities for the text it generated.
- **Tools.** There are no external tools. The only action a node has is writing text. That keeps the focus on how agents learn to communicate.
- **Control flow.** A feed-forward pass, layer by layer. Every node in a layer receives the problem and its private incoming messages, then writes its output. Non-final layers address messages to specific downstream nodes as `TO #k: ...`. The aggregator reads all outputs and answers.

### Key components

1. **Per-node LoRA adapters on a frozen backbone.** All agents share one copy of the model. Each agent owns only a small set of adapter weights, so adding an agent is cheap.
2. **Role-free, structure-aware prompts.** A node sees a header such as `[Network: 2-layer 1-1, 2 nodes | Layer 1/2, Position 1/1]`. It is never told to be a planner or a critic. Any specialization has to be learned.
3. **Section-routed messages.** A small parser splits `TO #1: ... TO #2: ...` into one private message per recipient. Routing is deterministic, so all learning happens inside the nodes.
4. **Joint REINFORCE.** Every node receives the same terminal reward $r \in \{0, 1\}$. Node $v$ minimizes its own surrogate loss

$$
\mathcal{L}_v = -\,(r - b_v)\,\bar{\ell}_v,
$$

where $\bar{\ell}_v$ is the mean log-probability of the tokens that node generated, and $b_v$ is that node's exponential moving average of past rewards. Subtracting the baseline reduces variance without changing the direction of the gradient in expectation. Using the mean rather than the sum keeps the loss from depending on message length.

5. **Progressive growth.** Train the small system first. Then expand the topology: inherited nodes keep their trained adapters and are held fixed during the next stage, new nodes start with inactive adapters, and training continues under the same reward.

### Architecture

![NeuroMAS overview](../images/neuromas_overview.png)

The naming convention counts LLM calls per query:

| System | Topology | Nodes | LLM calls |
|---|---|---|---|
| NeuroMAS-3 | `[1, 1]` + aggregator | 3 | 3 |
| NeuroMAS-5 | `[2, 2]` + aggregator | 5 | 5 |
| NeuroMAS-7 | `[2, 2, 2]` + aggregator | 7 | 7 |

## Required Packages

Training needs a CUDA GPU (Google Colab works). No API keys are required: the model and dataset are pulled from the Hugging Face Hub.

In [ ]:
!pip install -q transformers==4.51.3 peft==0.15.2 datasets==2.21.0 accelerate==1.6.0 tqdm

## Implementation

### Step 1: Imports and configuration

These are the settings used in the paper. Each one maps to a concept above: LoRA rank 8 per node, a learning rate of 2e-5, an EMA baseline with decay 0.9, and message truncation at 200 characters.

In [ ]:
import gc
import random
import re
from dataclasses import dataclass

import torch
import torch.optim as optim
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer


class Config:
    """Hyperparameters for NeuroMAS training and evaluation."""

    model_name = "Qwen/Qwen3-0.6B"
    seed = 42

    # One LoRA adapter per node.
    lora_rank = 8
    lora_alpha = 16
    lora_dropout = 0.05
    lora_targets = ["q_proj", "k_proj", "v_proj", "o_proj"]

    # REINFORCE with a per-node EMA reward baseline.
    learning_rate = 2e-5
    grad_clip = 1.0
    ema_decay = 0.9

    # Generation budgets and message truncation.
    inner_tokens = 250  # non-final node
    outer_tokens = 150  # output aggregation node
    section_max_chars = 200

    # Data and checkpoint selection.
    train_size = 300
    dev_size = 30
    eval_size = 200
    checkpoint_every = 50
    dev_eval_n = 30

    # RL steps for NeuroMAS-3, then growth to -5 and -7.
    steps = (1000, 800, 800)

### Step 2: Choose the run size

`QUICK_DEMO = True` runs a short walkthrough so you can see every part of the system work in a few minutes. Set it to `False` to use the full training budget from the paper.

In [ ]:
QUICK_DEMO = True

if QUICK_DEMO:
    Config.steps = (40, 20, 20)
    Config.train_size = 40
    Config.dev_size = 8
    Config.dev_eval_n = 8
    Config.eval_size = 20
    Config.checkpoint_every = 20

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on {device}. Training requires a CUDA GPU.")


def seed_all():
    """Seed Python and PyTorch so data splits and adapter initialization repeat."""
    random.seed(Config.seed)
    torch.manual_seed(Config.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(Config.seed)

### Step 3: Backbone and generation

A node needs two generation modes. During evaluation it only has to produce text. During training it also has to report the log-probability of each token it produced, because REINFORCE needs those values to carry gradients. So `generate_with_log_probs` generates under `no_grad`, then runs one differentiable forward pass over the finished sequence.

In [ ]:
def load_backbone():
    """Load the shared backbone and its tokenizer."""
    tokenizer = AutoTokenizer.from_pretrained(Config.model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        Config.model_name, torch_dtype=torch.float16, device_map=device
    )
    return model, tokenizer


def generate_text(model, tokenizer, prompt, max_new_tokens):
    """Generate a continuation with greedy decoding and return it as text."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(device)
    prompt_length = inputs.input_ids.shape[1]
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id
        )
    return tokenizer.decode(output[0][prompt_length:], skip_special_tokens=True)


def generate_with_log_probs(model, tokenizer, prompt, max_new_tokens):
    """Generate text and return it with the differentiable log-probability of each token."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(device)
    prompt_length = inputs.input_ids.shape[1]
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id
        )
    generated_ids = output[0][prompt_length:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    # This forward pass runs with gradients on, so the log-probs can drive the update.
    with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
        logits = model(output[0].unsqueeze(0)).logits
    log_probs = []
    for position, token_id in enumerate(generated_ids):
        token_log_probs = torch.log_softmax(logits[0, prompt_length + position - 1, :], dim=-1)
        log_probs.append(token_log_probs[token_id])
    return text, (torch.stack(log_probs) if log_probs else torch.tensor([]))

### Step 4: One backbone, many agents

This is what makes the system lightweight. PEFT gives us one set of LoRA weights on the model. `AdapterManager` stores one copy of those weights for every node and swaps the active copy in place when it is that node's turn.

New adapters are initialized the way LoRA recommends: the `A` matrix gets random values and the `B` matrix gets zeros. Since the adapter's contribution is `B @ A`, a new node starts out behaving exactly like the backbone.

In [ ]:
class AdapterManager:
    """Keep one LoRA state per node and swap the active state into the model."""

    def __init__(self, peft_model, num_nodes):
        self.model = peft_model
        self.states = [self._read_active()]
        self.current = 0
        for _ in range(num_nodes - 1):
            for name, parameter in self.model.named_parameters():
                if "lora_" in name and parameter.requires_grad:
                    if "lora_A" in name:
                        torch.nn.init.kaiming_uniform_(parameter, a=5**0.5)
                    elif "lora_B" in name:
                        torch.nn.init.zeros_(parameter)
            self.states.append(self._read_active())

    def _read_active(self):
        """Copy the trainable weights currently in the model."""
        return {n: p.data.clone() for n, p in self.model.named_parameters() if p.requires_grad}

    def _write_active(self, state):
        """Copy a stored state into the model."""
        for name, parameter in self.model.named_parameters():
            if name in state:
                parameter.data.copy_(state[name])

    def switch(self, node_index):
        """Make `node_index` the active agent, saving the previous agent's weights first."""
        if node_index != self.current:
            self.states[self.current] = self._read_active()
            self._write_active(self.states[node_index])
            self.current = node_index

    def save(self):
        """Return a CPU snapshot of every node's adapter."""
        self.states[self.current] = self._read_active()
        return [{k: v.clone().cpu() for k, v in state.items()} for state in self.states]

    def load(self, saved):
        """Restore a snapshot and make node 0 active."""
        for index, state in enumerate(saved):
            self.states[index] = {k: v.to(device) for k, v in state.items()}
        self._write_active(self.states[0])
        self.current = 0

    def get_state(self, node_index):
        """Return a copy of one node's adapter."""
        if node_index == self.current:
            return self._read_active()
        return {k: v.clone() for k, v in self.states[node_index].items()}


def attach_lora(backbone):
    """Wrap the backbone with LoRA, which also freezes all backbone weights."""
    return get_peft_model(
        backbone,
        LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            r=Config.lora_rank,
            lora_alpha=Config.lora_alpha,
            lora_dropout=Config.lora_dropout,
            target_modules=Config.lora_targets,
            bias="none",
        ),
    )

### Step 5: Structure-aware prompts and private routing

Read these prompts closely, because they carry a central idea of the method: **no prompt mentions a role**. A node learns only the shape of the network, where it sits, and what format its output must take.

`parse_sections` is the network's wiring. If a node writes `TO #2: check the second step`, only downstream node 2 receives that text. If a node does not follow the format, the parser falls back to sending its whole output to everyone, so a pass never breaks.

In [ ]:
SECTION_PATTERN = re.compile(r"TO\s*#\s*(\d+)\s*[:：]", re.IGNORECASE)


def parse_sections(text, num_recipients):
    """Split a node's output into one private message per downstream node."""
    max_chars = Config.section_max_chars
    fallback = text.strip()[:max_chars] if text.strip() else "(empty)"
    matches = list(SECTION_PATTERN.finditer(text))
    if not matches:
        return [fallback] * num_recipients

    sections = [None] * num_recipients
    for index, match in enumerate(matches):
        recipient = int(match.group(1)) - 1
        if not 0 <= recipient < num_recipients:
            continue
        end = matches[index + 1].start() if index + 1 < len(matches) else len(text)
        if sections[recipient] is None:
            sections[recipient] = text[match.end():end].strip()[:max_chars]
    return [section if section is not None else fallback for section in sections]


def node_position(node_index, topology):
    """Convert a flat node index into (layer, position)."""
    cumulative = 0
    for layer, width in enumerate(topology):
        if node_index < cumulative + width:
            return layer, node_index - cumulative
        cumulative += width
    raise ValueError(f"node {node_index} out of range for topology {topology}")


def node_prompt(task, topology, layer, position, question, incoming):
    """Build the role-free, structure-aware prompt for one node."""
    num_layers = len(topology)
    header = (
        f"[Network: {num_layers}-layer {'-'.join(map(str, topology))}, {sum(topology)} nodes | "
        f"Layer {layer + 1}/{num_layers}, Position {position + 1}/{topology[layer]}]"
    )
    parts = [header, f"\nPROBLEM:\n{question}"]
    if incoming:
        previous = "\n".join(f"[from L{l + 1}P{p + 1}]: {message}" for l, p, message in incoming)
        parts.append(f"\nPREVIOUS:\n{previous}")
    if layer == num_layers - 1:
        parts.append(f"\n{task.answer_format}")
    else:
        recipients = topology[layer + 1]
        message_format = "\n".join(f"TO #{k + 1}: <message for downstream node {k + 1}>" for k in range(recipients))
        parts.append(f"\nWrite a private message to each downstream node:\n{message_format}")
    return "\n".join(parts)


def aggregator_prompt(task, topology, question, all_outputs):
    """Build the prompt for the output aggregation node."""
    header = f"[Summarizer | Network: {len(topology)}-layer {'-'.join(map(str, topology))}, {sum(topology)} nodes]"
    return f"{header}\n\nPROBLEM:\n{question}\n\nALL OUTPUTS:\n{all_outputs}\n{task.answer_format}"

### Step 6: The forward pass

This is the network's `forward`. Nodes run layer by layer, and for each one we swap in its adapter, build its prompt from its private incoming messages, and generate. Frozen nodes generate but record no log-probabilities, so they are not updated. The aggregator goes last and produces the answer.

In [ ]:
def forward_pass(task, topology, model, tokenizer, adapters, question, training=True, frozen_nodes=frozenset()):
    """Run one query through the network and return node outputs and the final answer."""
    aggregator_index = sum(topology)
    layer_outputs, all_outputs = [], []
    node_index = 0

    for layer, width in enumerate(topology):
        is_last_layer = layer == len(topology) - 1
        current_layer = []
        for position in range(width):
            incoming = []
            if layer > 0:
                for sender in layer_outputs[layer - 1]:
                    sender_layer, sender_position = node_position(sender["node"], topology)
                    if sender["sections"] and position < len(sender["sections"]):
                        incoming.append((sender_layer, sender_position, sender["sections"][position]))

            adapters.switch(node_index)
            model.train() if training else model.eval()
            prompt = node_prompt(task, topology, layer, position, question, incoming)
            budget = Config.outer_tokens if is_last_layer else Config.inner_tokens
            if training and node_index not in frozen_nodes:
                text, log_probs = generate_with_log_probs(model, tokenizer, prompt, budget)
            else:
                text, log_probs = generate_text(model, tokenizer, prompt, budget), None

            sections = None if is_last_layer else parse_sections(text, topology[layer + 1])
            current_layer.append({"node": node_index, "text": text, "sections": sections})
            all_outputs.append((node_index, text, log_probs))
            node_index += 1
        layer_outputs.append(current_layer)

    summary = "\n".join(f"[N{node + 1}]: {text[:250]}" for node, text, _ in all_outputs)
    adapters.switch(aggregator_index)
    model.train() if training else model.eval()
    prompt = aggregator_prompt(task, topology, question, summary)
    if training:
        answer, answer_log_probs = generate_with_log_probs(model, tokenizer, prompt, Config.outer_tokens)
    else:
        answer, answer_log_probs = generate_text(model, tokenizer, prompt, Config.outer_tokens), None
    return all_outputs, answer, answer_log_probs

### Step 7: Joint training from one terminal reward

Here is the entire learning rule. After one forward pass we score the final answer once. Then every node, including the aggregator, takes a gradient step using that same reward, its own baseline, and its own log-probabilities.

Nothing here tells a node what it did well. A node that wrote a useful message and a node that wrote a useless one receive the same reward on a given query. Over many queries, the messages that tend to lead to correct answers are reinforced. This is why the training loop is so short.

Every `checkpoint_every` steps we measure accuracy on a held-out dev set and remember the best adapters.

In [ ]:
def update_node(model, parameters, optimizer, log_probs, reward, baseline):
    """Apply one REINFORCE step for the active node."""
    loss = -(log_probs.mean() * (reward - baseline))
    if not torch.isnan(loss):
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(parameters, Config.grad_clip)
        optimizer.step()


def train_stage(task, topology, model, tokenizer, adapters, train_set, dev_set, steps, frozen_nodes=frozenset(), label=""):
    """Train one topology with REINFORCE and restore its best dev checkpoint."""
    aggregator_index = sum(topology)
    parameters = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(parameters, lr=Config.learning_rate)
    baselines = [0.0] * (aggregator_index + 1)
    decay = Config.ema_decay
    dev_subset = dev_set[: Config.dev_eval_n]
    best_dev, best_states = -1.0, None

    for step in tqdm(range(steps), desc=f"Training {label}"):
        torch.cuda.empty_cache()
        example = train_set[step % len(train_set)]
        outputs, answer, answer_log_probs = forward_pass(
            task, topology, model, tokenizer, adapters, example.question, training=True, frozen_nodes=frozen_nodes
        )

        # The one reward that every node shares.
        reward = 1.0 if task.score(answer, example) else 0.0

        baselines[aggregator_index] = decay * baselines[aggregator_index] + (1 - decay) * reward
        if answer_log_probs is not None and len(answer_log_probs) > 0:
            adapters.switch(aggregator_index)
            update_node(model, parameters, optimizer, answer_log_probs, reward, baselines[aggregator_index])

        for node, _, log_probs in outputs:
            baselines[node] = decay * baselines[node] + (1 - decay) * reward
            if node in frozen_nodes:
                continue
            if log_probs is not None and len(log_probs) > 0:
                adapters.switch(node)
                update_node(model, parameters, optimizer, log_probs, reward, baselines[node])

        if (step + 1) % Config.checkpoint_every == 0:
            dev_accuracy = evaluate(task, topology, model, tokenizer, adapters, dev_subset, show_progress=False)
            if dev_accuracy > best_dev:
                best_dev, best_states = dev_accuracy, adapters.save()

    if best_states is not None:
        adapters.load(best_states)
    return adapters


def evaluate(task, topology, model, tokenizer, adapters, examples, show_progress=True):
    """Return the accuracy of the network on `examples`."""
    correct = 0
    for example in tqdm(examples, desc=f"Evaluating {sum(topology) + 1} agents", disable=not show_progress):
        _, answer, _ = forward_pass(task, topology, model, tokenizer, adapters, example.question, training=False)
        correct += int(task.score(answer, example))
    return correct / max(1, len(examples))

### Step 8: Progressive growth

Growth maps old node indices to their positions in the larger topology. Going from NeuroMAS-3 to NeuroMAS-5, the first-layer node stays at index 0, the second-layer node moves to index 2 (the first slot of the new second layer), and the aggregator moves to index 4. Inherited worker nodes are held fixed while the new nodes learn to work with them.

In [ ]:
GROW_3_TO_5 = {0: 0, 1: 2, 2: 4}
GROW_5_TO_7 = {0: 0, 1: 1, 2: 2, 3: 3, 4: 6}


def grow_adapters(model, new_size, old_adapters, mapping):
    """Create adapters for a larger network and copy trained nodes into their new slots."""
    new_adapters = AdapterManager(model, new_size)
    for old_index, new_index in mapping.items():
        new_adapters.states[new_index] = {k: v.to(device) for k, v in old_adapters.get_state(old_index).items()}
    new_adapters._write_active(new_adapters.states[0])
    new_adapters.current = 0
    return new_adapters


def fresh_model():
    """Release the previous model and load a clean backbone with LoRA attached."""
    gc.collect()
    torch.cuda.empty_cache()
    backbone, tokenizer = load_backbone()
    return attach_lora(backbone), tokenizer


def run_neuromas(task, up_to=3):
    """Train NeuroMAS with progressive growth and return accuracy for each stage."""
    train_set, dev_set, eval_set = task.load()
    results = {}

    model, tokenizer = fresh_model()
    adapters = AdapterManager(model, 3)
    adapters = train_stage(task, [1, 1], model, tokenizer, adapters, train_set, dev_set, Config.steps[0], label="NeuroMAS-3")
    results["NeuroMAS-3"] = evaluate(task, [1, 1], model, tokenizer, adapters, eval_set)
    saved = adapters.save()
    if up_to <= 3:
        return results, model, tokenizer, adapters, eval_set

    del model
    model, tokenizer = fresh_model()
    restored = AdapterManager(model, 3)
    restored.load(saved)
    adapters = grow_adapters(model, 5, restored, GROW_3_TO_5)
    adapters = train_stage(
        task, [2, 2], model, tokenizer, adapters, train_set, dev_set, Config.steps[1], frozen_nodes={0, 2}, label="NeuroMAS-5"
    )
    results["NeuroMAS-5"] = evaluate(task, [2, 2], model, tokenizer, adapters, eval_set)
    saved = adapters.save()
    if up_to <= 5:
        return results, model, tokenizer, adapters, eval_set

    del model
    model, tokenizer = fresh_model()
    restored = AdapterManager(model, 5)
    restored.load(saved)
    adapters = grow_adapters(model, 7, restored, GROW_5_TO_7)
    adapters = train_stage(
        task, [2, 2, 2], model, tokenizer, adapters, train_set, dev_set, Config.steps[2], frozen_nodes={0, 1}, label="NeuroMAS-7"
    )
    results["NeuroMAS-7"] = evaluate(task, [2, 2, 2], model, tokenizer, adapters, eval_set)
    return results, model, tokenizer, adapters, eval_set

### Step 9: The task

We use **Navigate** from BIG-Bench Hard, a spatial reasoning benchmark: given a series of moves, does the walker end up back at the start? The task object has only three jobs: load the data, tell nodes the answer format, and turn a final answer into a reward.

In [ ]:
@dataclass
class Example:
    """One benchmark question and its gold answer."""

    id: str
    question: str
    answer: str


def parse_yes_no(text):
    """Return whichever of yes or no appears first in the text."""
    if not text:
        return None
    lowered = text.lower()
    candidates = [(index, value) for index, value in [(lowered.find("yes"), "yes"), (lowered.find("no"), "no")] if index != -1]
    return min(candidates)[1] if candidates else None


class NavigateTask:
    """BIG-Bench Hard Navigate with a binary terminal reward."""

    name = "navigate"
    answer_format = "Answer Yes or No.\nAnswer:"

    def load(self):
        """Return disjoint train, dev, and evaluation splits."""
        seed_all()
        dataset = load_dataset("lukaemon/bbh", "navigate", split="test", trust_remote_code=True)
        pool = [
            Example(f"nav_{i}", dataset[i]["input"], dataset[i]["target"].strip().lower())
            for i in range(len(dataset))
        ]
        random.shuffle(pool)
        eval_count = min(200, Config.eval_size)
        eval_set = pool[:eval_count]

        eval_questions = {example.question for example in eval_set}
        seen, remaining = set(), []
        for example in pool[eval_count:]:
            if example.question in eval_questions or example.question in seen:
                continue
            seen.add(example.question)
            remaining.append(example)
        dev_count = min(Config.dev_size, max(1, len(remaining) // 3))
        return remaining[dev_count:], remaining[:dev_count], eval_set

    def score(self, answer, example):
        """Return True when the final answer matches the gold label."""
        prediction = parse_yes_no(answer)
        return prediction is not None and prediction == example.answer


task = NavigateTask()

## Usage Example

### Train NeuroMAS-3

Passing `up_to=5` or `up_to=7` runs the full growth chain and reports every stage.

In [ ]:
seed_all()
results, model, tokenizer, adapters, eval_set = run_neuromas(task, up_to=3)
for stage, accuracy in results.items():
    print(f"{stage}: {accuracy * 100:.1f}%")

### Watch the agents talk

Run one query through the trained network and print what each node wrote. This view is the best way to build intuition: you can see the upstream message, what the next node did with it, and how the aggregator turned that into an answer. On a full training run, compare these traces with an untrained network.

In [ ]:
example = eval_set[0]
outputs, answer, _ = forward_pass(task, [1, 1], model, tokenizer, adapters, example.question, training=False)

print("QUESTION:", example.question, "\n")
for node, text, _ in outputs:
    print(f"--- Node {node + 1} ---\n{text}\n")
print("--- Aggregator ---\n", answer)
print("\nGold answer:", example.answer, "| Reward:", int(task.score(answer, example)))

## Comparison

### Quantitative: one direct call to the same backbone

This cell measures the simplest possible agent, a single direct prompt to the untrained backbone, on the same evaluation questions.

In [ ]:
def direct_accuracy(task, examples):
    """Measure a single direct prompt to the backbone without any adapter."""
    backbone, base_tokenizer = load_backbone()
    backbone.eval()
    correct = 0
    for example in tqdm(examples, desc="Direct prompting"):
        prompt = f"{example.question}\n{task.answer_format}"
        correct += int(task.score(generate_text(backbone, base_tokenizer, prompt, Config.outer_tokens), example))
    return correct / max(1, len(examples))


print(f"Direct prompting: {direct_accuracy(task, eval_set) * 100:.1f}%")

### Results from the paper

With the full training budget and Qwen3-0.6B (6.9M trainable parameters for NeuroMAS-3 and for the matched controls), the paper reports the following accuracy (%):

| Method | ARC | Navigate | Algebra | Physics | Medicine | HumanEval | LLM calls |
|---|---|---|---|---|---|---|---|
| Direct prompting | 24.5 | 40.5 | 22.0 | 17.6 | 24.0 | 11.0 | 1 |
| GPTSwarm (fixed backbone) | 46.0 | 40.5 | 25.0 | 25.5 | 43.5 | 12.8 | 11 |
| Single-LLM RL (same parameter budget) | 46.0 | 42.0 | 29.0 | 24.5 | 43.0 | 17.7 | 1 |
| **NeuroMAS-3** | **56.5** | **45.5** | **39.0** | **44.1** | **48.0** | **30.5** | **3** |
| **NeuroMAS-7** | 53.5 | **51.0** | **42.0** | 39.2 | 43.5 | **31.7** | 7 |

NeuroMAS-3 and the single-agent RL control share the same backbone, objective, and number of trainable parameters. The gain comes from how those parameters are organized into communicating agents.

### Qualitative

| | Single agent | Hand-designed multi-agent | NeuroMAS |
|---|---|---|---|
| Who decides each agent's job? | Not applicable | The developer | Training |
| Training signal | Final reward | Often none, or role-specific rewards | One final reward |
| Cost of adding an agent | Not applicable | New prompts and wiring | One LoRA adapter |
| Scaling | Bigger model | Redesign the workflow | Grow the topology |

## Additional Considerations

**Use cases.** NeuroMAS suits settings with a checkable outcome and a small model budget, such as multiple-choice reasoning, verifiable math, and code that can be unit tested. Because all agents share one backbone, a 0.6B model is enough to train a whole team on a single GPU.

**Hardware.** A single GPU is enough for the paper's setup, since only one backbone is loaded and adapters are swapped in place.

**Ideas to try.**

- Replace the reward with your own verifier. `task.score` is the only place the objective appears.
- Try another topology. `forward_pass` accepts any list of layer widths.
- Add a new benchmark by writing a task with `answer_format`, `load`, and `score`.
- Inspect message traces before and after training to see which communication patterns emerge.

The full reference implementation, with all six benchmarks, is at [haoranlustat/NeuroMAS](https://github.com/haoranlustat/NeuroMAS).

## References

- Lu, H., Fang, L., Zhong, W., and Ma, P. [NeuroMAS: Multi-Agent Systems as Neural Networks with Joint Reinforcement Learning](https://arxiv.org/abs/2605.16757). *The First Conference on Statistics and Trustworthy AI for Cross (X)-Domain Acceleration (STAI-X 2026)*. Paper Award Honorable Mention.
- Reference code: [github.com/haoranlustat/NeuroMAS](https://github.com/haoranlustat/NeuroMAS)
- Williams, R. J. (1992). Simple statistical gradient-following algorithms for connectionist reinforcement learning. *Machine Learning*.
- Hu, E. J. et al. (2022). [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685). *ICLR*.